# Data Understanding

This notebook loads the raw dataset, performs an initial structure review, and records early observations before preprocessing and feature engineering.

In [1]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("data/raw/Liberty_Heritage_Data.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../data/raw/Liberty_Heritage_Data.csv")

df = pd.read_csv(DATA_PATH)

print(f"Loaded {DATA_PATH} with shape {df.shape}")
print(df.head().to_string(index=False))

Loaded ../data/raw/Liberty_Heritage_Data.csv with shape (8000, 25)
ApplicationID  Age Gender MaritalStatus  Dependents EducationLevel State       ResidenceType  YearsAtCurrentResidence      EmploymentType  Employed  EmploymentLengthYears  AnnualIncome  IncomeVerified  CreditScore  CreditHistoryMonths  ExistingLoanAccounts  ExistingCreditCards  TotalMonthlyDebtPayment  DebtToIncomeRatio  RevolvingUtilization  PriorDefault  BankruptcyLast7Years  EnquiriesLast6Months Approval
  LHB-2403641   60 Female      Divorced           4     Bachelor's    GA                Rent                      2.0 Salaried-Government         1                    2.0         100.2               1          513                  369                     4                    3                   2271.0              0.272                 0.484             1                     0                     3       No
  LHB-2405086   28   Male       Married           1      Doctorate    KY Own (with mortgage)                   

## Structured Summary

In [2]:
summary = {
    "shape": df.shape,
    "columns": list(df.columns),
    "dtypes": df.dtypes.astype(str).to_dict(),
    "missing_values": df.isna().sum().to_dict(),
    "duplicate_rows": int(df.duplicated().sum()),
}

print(summary)

{'shape': (8000, 25), 'columns': ['ApplicationID', 'Age', 'Gender', 'MaritalStatus', 'Dependents', 'EducationLevel', 'State', 'ResidenceType', 'YearsAtCurrentResidence', 'EmploymentType', 'Employed', 'EmploymentLengthYears', 'AnnualIncome', 'IncomeVerified', 'CreditScore', 'CreditHistoryMonths', 'ExistingLoanAccounts', 'ExistingCreditCards', 'TotalMonthlyDebtPayment', 'DebtToIncomeRatio', 'RevolvingUtilization', 'PriorDefault', 'BankruptcyLast7Years', 'EnquiriesLast6Months', 'Approval'], 'dtypes': {'ApplicationID': 'str', 'Age': 'int64', 'Gender': 'str', 'MaritalStatus': 'str', 'Dependents': 'int64', 'EducationLevel': 'str', 'State': 'str', 'ResidenceType': 'str', 'YearsAtCurrentResidence': 'float64', 'EmploymentType': 'str', 'Employed': 'int64', 'EmploymentLengthYears': 'float64', 'AnnualIncome': 'float64', 'IncomeVerified': 'int64', 'CreditScore': 'int64', 'CreditHistoryMonths': 'int64', 'ExistingLoanAccounts': 'int64', 'ExistingCreditCards': 'int64', 'TotalMonthlyDebtPayment': 'floa

## Credit Score Checks

This section validates the bank's stated score and age rules, and measures how much of the portfolio falls in the target FICO band.

In [3]:
credit_score_min = int(df['CreditScore'].min())
credit_score_max = int(df['CreditScore'].max())
credit_score_within_range = df['CreditScore'].between(300, 850, inclusive='both').all()

age_min = int(df['Age'].min())
age_starts_at_18 = df['Age'].ge(18).all()

target_band_mask = df['CreditScore'].between(640, 780, inclusive='both')
target_band_count = int(target_band_mask.sum())
target_band_pct = round(target_band_mask.mean() * 100, 2)

credit_score_checks = {
    'credit_score_min': credit_score_min,
    'credit_score_max': credit_score_max,
    'credit_score_within_300_850': credit_score_within_range,
    'age_min': age_min,
    'age_at_least_18': age_starts_at_18,
    'target_band_count_640_780': target_band_count,
    'target_band_pct_640_780': target_band_pct,
}

print(credit_score_checks)
print(f"CreditScore range observed: {credit_score_min} to {credit_score_max}")
print(f"Age minimum observed: {age_min}")
print(f"Applicants with CreditScore between 640 and 780: {target_band_count} ({target_band_pct}%)")

{'credit_score_min': 342, 'credit_score_max': 837, 'credit_score_within_300_850': np.True_, 'age_min': 18, 'age_at_least_18': np.True_, 'target_band_count_640_780': 3451, 'target_band_pct_640_780': np.float64(43.14)}
CreditScore range observed: 342 to 837
Age minimum observed: 18
Applicants with CreditScore between 640 and 780: 3451 (43.14%)


In [4]:
numeric_cols = df.select_dtypes(include="number").columns
categorical_cols = df.select_dtypes(exclude="number").columns

numeric_ranges = (
    df[numeric_cols]
    .agg(["min", "max", "mean", "median"])
    .T
    .rename_axis("column")
)

categorical_summary = pd.DataFrame(
    {
        "unique_values": [df[col].nunique(dropna=True) for col in categorical_cols],
        "values": [sorted(df[col].dropna().astype(str).unique().tolist()) for col in categorical_cols],
    },
    index=categorical_cols,
)

print("Numeric value ranges")
print(numeric_ranges.to_string())

print("Categorical columns")
print(categorical_summary.to_string())

Numeric value ranges
                            min       max         mean    median
column                                                          
Age                       18.00    78.000    40.053875    40.000
Dependents                 0.00     6.000     1.232500     1.000
YearsAtCurrentResidence    0.00    40.000     5.593370     4.000
Employed                   0.00     1.000     0.887875     1.000
EmploymentLengthYears      0.00    32.000     3.558442     2.000
AnnualIncome              12.00   410.000    69.696832    62.400
IncomeVerified             0.00     1.000     0.841375     1.000
CreditScore              342.00   837.000   628.215125   630.000
CreditHistoryMonths        0.00   578.000   198.430125   196.000
ExistingLoanAccounts       0.00     5.000     1.371125     1.000
ExistingCreditCards        0.00     7.000     2.395875     2.000
TotalMonthlyDebtPayment   39.00  6910.000  1406.952375  1276.500
DebtToIncomeRatio          0.02     2.000     0.321237     0.242
Revo

## Column Descriptions

| Column | Type | Description |
|---|---|---|
| ApplicationID | categorical | Unique application identifier |
| Age | numeric | Applicant age in years |
| Gender | categorical | Self-reported gender |
| MaritalStatus | categorical | Applicant marital status |
| Dependents | numeric | Number of dependents |
| EducationLevel | categorical | Highest education level |
| State | categorical | US state code |
| ResidenceType | categorical | Housing situation |
| YearsAtCurrentResidence | numeric | Years at current residence |
| EmploymentType | categorical | Employment category |
| Employed | numeric/binary | Employment flag |
| EmploymentLengthYears | numeric | Length of current employment in years |
| AnnualIncome | numeric | Annual income |
| IncomeVerified | numeric/binary | Income verification flag |
| CreditScore | numeric | Credit score |
| CreditHistoryMonths | numeric | Length of credit history in months |
| ExistingLoanAccounts | numeric | Number of existing loan accounts |
| ExistingCreditCards | numeric | Number of existing credit cards |
| TotalMonthlyDebtPayment | numeric | Total monthly debt payment |
| DebtToIncomeRatio | numeric | Debt-to-income ratio |
| RevolvingUtilization | numeric | Revolving utilization ratio |
| PriorDefault | numeric/binary | Prior default flag |
| BankruptcyLast7Years | numeric/binary | Bankruptcy flag in the last 7 years |
| EnquiriesLast6Months | numeric | Number of credit enquiries in the last 6 months |
| Approval | categorical/target | Loan approval outcome |


## Initial Observations

- The raw file has 8,000 rows and 25 columns, with 7 categorical fields and 18 numeric/binary fields.
- There are 541 missing values overall, but no duplicate rows in the file.
- `ApplicationID` is unique across all rows, so it is best treated as an identifier rather than a predictive feature.
- `Approval` is the likely target column, and the class balance should be checked before modeling.
- Several binary variables are stored as integers, so they will need to be interpreted carefully when engineering features.
- `RevolvingUtilization` exceeds 1.0 for at least some records, which suggests either over-utilization cases or values that warrant validation.
- The numeric features are on very different scales, so scaling may help some downstream models even though decision trees do not require it.